In [ ]:
from moabb.datasets import PhysionetMI
from moabb.paradigms import MotorImagery
from moabb.evaluations import WithinSessionEvaluation
from moabb.datasets.utils import find_intersecting_channels

from sklearn.pipeline import make_pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

from mne.decoding import CSP
from brainbot_dataset import get_brainbot_dataset
from physionet16 import PhysionetMI16
from custom_models.cnn import CNN

import moabb
import mne

moabb.set_log_level('ERROR')
mne.set_log_level('ERROR')

SUBJECTS = 1
MAX_TRIALS = 1

brainbot_dataset = get_brainbot_dataset()
brainbot_dataset.n_sessions = min(MAX_TRIALS, brainbot_dataset.n_sessions)
brainbot_dataset.subject_list = brainbot_dataset.subject_list[:SUBJECTS]
physionet_dataset = PhysionetMI()
physionet_dataset.subject_list = physionet_dataset.subject_list[:SUBJECTS]
physionet16_dataset = PhysionetMI16()
physionet16_dataset.subject_list = physionet16_dataset.subject_list[:SUBJECTS]
assert len(physionet16_dataset.subject_list) == SUBJECTS
assert len(physionet_dataset.subject_list) == SUBJECTS
assert len(brainbot_dataset.subject_list) == SUBJECTS
assert brainbot_dataset.n_sessions == MAX_TRIALS


datasets = [brainbot_dataset, physionet16_dataset, physionet_dataset]
dataset_results = {}
dataset_events = ["left_hand", "right_hand", "feet", "hands", "rest"]
sampling = 160 # based on Physionet sampling rate 

electrodes, datasets = find_intersecting_channels(datasets)
print("Datasets used:", [type(d).__name__ for d in datasets])
print("Used electrodes:", electrodes)

paradigm = MotorImagery(n_classes=len(dataset_events), events=dataset_events, resample=sampling)

pipelines = {}

# Base classifiers and preprocessing
svm = OneVsRestClassifier(SVC(kernel='rbf', probability=True))
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)

pipelines['CSP + SVM'] = make_pipeline(csp, svm)
pipelines['CSP + LDA'] = make_pipeline(CSP(n_components=8), LinearDiscriminantAnalysis())

# TGSP (Riemannian) pipeline
pipelines['TGSP + SVM'] = make_pipeline(Covariances("oas"), TangentSpace(metric="riemann"), SVC(kernel="linear", probability=True))

# Custom CNN pipeline
pipelines['CNN'] = CNN(sfreq=sampling)

evaluation = WithinSessionEvaluation(paradigm=paradigm, datasets=datasets, overwrite=True, n_jobs=-1)
results = evaluation.process(pipelines)

In [ ]:
print("Results Summary:")
summary = results.groupby(['pipeline', 'dataset'])['score'].agg(['mean', 'std', 'count'])
summary['mean'] = summary['mean'].round(3)
summary['std'] = summary['std'].round(3)
print(summary.to_string())
print("=" * 50)

print("\nDetailed Results by Subject and Dataset:")
detailed = results.pivot_table(
    index=['dataset', 'subject', 'session'], 
    columns='pipeline', 
    values='score'
)
print(detailed.round(3).to_string())
print("=" * 50)

### Test all builtin pipelines

In [ ]:
from moabb import benchmark
import os

moabb_pipelines_path = os.path.join(os.path.dirname(os.path.dirname(moabb.__file__)), "pipelines")
print(moabb_pipelines_path)

results = benchmark(
    pipelines=moabb_pipelines_path,
    evaluations=["WithinSession"],
    paradigms=["MotorImagery"],
    include_datasets=datasets,
    results="./results/",
    overwrite=True,
    plot=True,
    output="./benchmark/",
    n_jobs=-1,
)

In [ ]:
from moabb.analysis.results import Results
from moabb.evaluations import WithinSessionEvaluation
from moabb.paradigms import MotorImagery

# Load existing results  
results = Results(  
    evaluation_class=WithinSessionEvaluation,  
    paradigm_class=MotorImagery,  
    hdf5_path="./results"
)

# Convert to DataFrame
df = results.to_dataframe()  


print("Results Summary:")
summary = df.groupby(['pipeline', 'dataset'])['score'].agg(['mean', 'std', 'count']).reset_index()
summary['pipeline_max_mean'] = summary.groupby('pipeline')['mean'].transform('max')
summary = (summary.sort_values(['pipeline_max_mean', 'mean'], ascending=[False, False])
           .drop(columns='pipeline_max_mean')
           .set_index(['pipeline', 'dataset']))
summary['mean'] = summary['mean'].round(3)
summary['std'] = summary['std'].round(3)
print(summary.to_string())
print("=" * 50)

print("\nDetailed Results by Subject and Dataset:")
detailed = df.pivot_table(
    index=['dataset', 'subject', 'session'], 
    columns='pipeline', 
    values='score'
)
print(detailed.round(3).to_string())
print("=" * 50)